# 📈 Pronóstico de Series de Tiempo: Precio Residencial por $m^2$ en la ZMVM (E1)

**Proyecto Integrador — Trusted Data Lab (Etapa E1)**  
**Integrantes:** Juan Antonio Ortiz Hinojosa & Diego Alberto Virgen Cuéllar

---

## 🎯 Contenido del Notebook
1. **Descomposición Aditiva y Multiplicativa** de la serie histórica de 36 meses (`2023-11` a `2026-10`).
2. **Prueba de Estacionariedad Augmented Dickey-Fuller (ADF)** en nivel y primera diferencia ($\Delta Y_t$) + **ACF y PACF**.
3. **Entrenamiento y Backtesting de 4 Modelos:** Suavizamiento Exponencial Simple (SES), Método Lineal de Holt, Holt-Winters Estacional y ARIMA/SARIMA.
4. **Comparación en Conjunto de Prueba (MAE, RMSE, MAPE)** y **Diagnóstico de Residuos Ljung-Box**.
5. **Pronóstico a 4 Trimestres (12 meses)** con intervalos de confianza al 95%.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / "src").exists():
    ROOT_DIR = ROOT_DIR.parent.parent
sys.path.insert(0, str(ROOT_DIR))

from src.analytics_e3_e4 import run_time_series_forecasting_e1

res_e1 = run_time_series_forecasting_e1(horizon_months=12)
print("Serie histórica cargada:", len(res_e1["historico_df"]), "periodos mensuales.")

C:\Users\Colibecas\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\Colibecas\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params


Serie histórica cargada: 36 periodos mensuales.


## 1. Descomposición Aditiva/Multiplicativa, Prueba Dickey-Fuller (ADF) y ACF/PACF

In [2]:
print("=== Prueba Augmented Dickey-Fuller (ADF) ===")
print(json.dumps(res_e1["adf_test"], indent=2))
print("\nACF (primeros 8 rezagos):", res_e1["acf_values"][:8])
print("PACF (primeros 8 rezagos):", res_e1["pacf_values"][:8])

df_decomp = pd.DataFrame({
    "periodo": res_e1["descomposicion"]["periodos"],
    "observado": res_e1["descomposicion"]["observado"],
    "tendencia_aditiva": res_e1["descomposicion"]["tendencia_aditiva"],
    "estacional_aditiva": res_e1["descomposicion"]["estacional_aditiva"],
    "residuo_aditivo": res_e1["descomposicion"]["residuo_aditivo"],
    "estacional_multiplicativa": res_e1["descomposicion"]["estacional_multiplicativa"],
})
display(df_decomp.head(12))

=== Prueba Augmented Dickey-Fuller (ADF) ===
{
  "nivel_stat": 0.3013,
  "nivel_pvalue": 0.9774,
  "nivel_estacionaria": false,
  "primera_diferencia_stat": -4.1432,
  "primera_diferencia_pvalue": 0.000821,
  "primera_diferencia_estacionaria": true
}

ACF (primeros 8 rezagos): [1.0, 0.8889, 0.7733, 0.6567, 0.5944, 0.5419, 0.4897, 0.4147]
PACF (primeros 8 rezagos): [1.0, 0.8889, -0.0801, -0.07, 0.1916, -0.0065, -0.0476, -0.0935]


,periodo,observado,tendencia_aditiva,estacional_aditiva,residuo_aditivo,estacional_multiplicativa
0,2023-11,323.18,321.70,-1.47,2.95,0.9961
1,2023-12,320.65,323.14,-3.10,0.61,0.9909
2,2024-01,325.78,324.57,3.25,-2.04,1.0094
3,2024-02,326.44,326.00,2.13,-1.69,1.0059
4,2024-03,326.83,327.44,-0.85,0.25,0.9976
5,2024-04,325.30,328.87,-5.29,1.73,0.9849
6,2024-05,329.88,330.42,-0.86,0.32,0.9976
7,2024-06,325.96,331.65,-4.96,-0.73,0.9858
8,2024-07,336.88,333.10,3.71,0.08,1.0106
9,2024-08,340.66,334.63,5.12,0.91,1.0145


## 2. Comparación de los 4 Modelos (SES, Holt, Holt-Winters, ARIMA/SARIMA) y Prueba Ljung-Box

In [3]:
df_metrics = pd.DataFrame(res_e1["metricas_comparativas_modelos"])
display(df_metrics)

,modelo,MAE_MXN,RMSE_MXN,MAPE_pct,Ljung_Box_pvalue,residuos_ruido_blanco
0,1. Exponencial Simple (SES),11.96,13.46,3.19,0.0005,False
1,2. Lineal de Holt,4.59,5.26,1.23,0.0022,False
2,3. Holt-Winters (Tendencia + Estacionalidad),2.54,3.29,0.68,0.2512,True
3,"4. ARIMA / SARIMA (1,1,1)",13.10,14.50,3.50,0.0088,False


## 3. Pronóstico a 12 Meses (4 Trimestres) con Intervalos de Confianza al 95%

In [4]:
print("=== Pronóstico Mensual (12 Meses) ===")
display(res_e1["pronostico_12_meses_df"])
print("\n=== Resumen Trimestral (Próximos 4 Trimestres) ===")
display(res_e1["pronostico_4_trimestres_df"])

=== Pronóstico Mensual (12 Meses) ===


,periodo_mensual,trimestre,pronostico_SES_mxn,pronostico_Holt_mxn,pronostico_HoltWinters_mxn,pronostico_SARIMA_mxn,ic95_inferior_mxn,ic95_superior_mxn
0,2026-11,2026-Q4,372.71,375.89,373.88,371.81,365.11,378.50
1,2026-12,2026-Q4,372.71,377.42,373.72,375.31,368.39,382.22
2,2027-01,2027-Q1,372.71,378.95,381.56,384.61,375.83,393.38
3,2027-02,2027-Q1,372.71,380.49,381.94,388.03,378.85,397.21
4,2027-03,2027-Q1,372.71,382.02,380.48,381.08,370.77,391.40
5,2027-04,2027-Q2,372.71,383.55,377.56,377.39,366.59,388.19
6,2027-05,2027-Q2,372.71,385.09,383.43,384.86,373.23,396.49
7,2027-06,2027-Q2,372.71,386.62,380.77,381.58,369.44,393.73
8,2027-07,2027-Q3,372.71,388.15,391.01,394.37,381.55,407.19
9,2027-08,2027-Q3,372.71,389.69,394.06,398.49,385.16,411.83



=== Resumen Trimestral (Próximos 4 Trimestres) ===


,trimestre,precio_m2_proyectado_SARIMA_mxn,precio_m2_proyectado_HoltWinters_mxn,ic95_inferior_mxn,ic95_superior_mxn
0,2026-Q4,373.56,373.80,366.75,380.36
1,2027-Q1,384.57,381.33,375.15,394.00
2,2027-Q2,381.28,380.59,369.75,392.80
3,2027-Q3,397.72,393.38,384.36,411.07
4,2027-Q4,391.54,389.82,377.13,405.95
